# **Final Model Notebook: ARMA Baseline, GARCH-family, True EGARCH-X, HAC, and Weekly SVAR**

- Main volatility model: ARMA(3,2) mean equation with residual-based GARCH-family and custom EGARCH-X volatility modeling.
- Supplementary explanation: HAC shock-volatility regression.
- Supplementary dynamic transmission: weekly SVAR with structural IRF.

In [28]:
# ============================================================
# 0. IMPORTS AND GLOBAL SETTINGS
# ============================================================

import sys
import subprocess
import importlib.util
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

if importlib.util.find_spec("arch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "arch"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.special import gammaln, expit, logit
from scipy.stats import norm

import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.svar_model import SVAR

from arch import arch_model

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

MAX_ARMA_P = 3
MAX_ARMA_Q = 3
RUN_ARMA_GRID = True

# EGARCH-X settings
RUN_TRUE_EGARCHX = True
EGARCHX_MAXITER = 3000
COMPUTE_EGARCHX_SE = False  

def find_existing_outputs_dir(start=Path.cwd()):
    start = Path(start).resolve()

    for p in [start] + list(start.parents):
        candidate = p / "outputs"
        if candidate.exists() and candidate.is_dir():
            return candidate

    raise FileNotFoundError(
        "Không tìm thấy folder 'outputs' có sẵn trong current directory hoặc parent directories."
    )

OUTPUTS_DIR = find_existing_outputs_dir()
OUT_DIR = OUTPUTS_DIR / "model_egarchx_hac_svar_final"
OUT_DIR.mkdir(exist_ok=True)

## 1. Load Data and Construct Missing Features

The notebook first tries to load `model_data.csv`, then `processed_data.csv`, then `dataset.csv`.
If return variables are missing, they are constructed automatically.

In [8]:
def find_first_existing_path(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

df = pd.read_csv("../data/processed/model_data.csv")

if "Date" not in df.columns:
    raise ValueError("Missing required Date column.")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)
df = df.replace([np.inf, -np.inf], np.nan)

print("Raw shape:", df.shape)
print("Date range:", df["Date"].min(), "→", df["Date"].max())
print("Columns:", list(df.columns))

if "usd_vnd" in df.columns and "log_usd_vnd" not in df.columns:
    df["log_usd_vnd"] = np.log(df["usd_vnd"].astype(float))

if "fx_return" not in df.columns:
    if "usd_vnd" not in df.columns:
        raise ValueError("Need either fx_return or usd_vnd to construct fx_return.")
    df["fx_return"] = 100 * np.log(df["usd_vnd"].astype(float)).diff()

if "abs_return" not in df.columns:
    df["abs_return"] = df["fx_return"].abs()

if "squared_return" not in df.columns:
    df["squared_return"] = df["fx_return"] ** 2

if "rolling_vol_22d" not in df.columns:
    df["rolling_vol_22d"] = df["fx_return"].rolling(window=22, min_periods=15).std()

if "vix" in df.columns and "vix_change" not in df.columns:
    df["vix_change"] = df["vix"].astype(float).diff()

if "us_10y" in df.columns and "us10y_change" not in df.columns:
    df["us10y_change"] = df["us_10y"].astype(float).diff()

if "dxy" in df.columns and "dxy_return" not in df.columns:
    df["dxy_return"] = 100 * np.log(df["dxy"].astype(float)).diff()

if "vnindex" in df.columns and "vnindex_return" not in df.columns:
    df["vnindex_return"] = 100 * np.log(df["vnindex"].astype(float)).diff()

oil_price_candidates = ["wti_oil", "DCOILWTICO", "oil", "oil_price", "crude_oil"]
oil_price_col = next((c for c in oil_price_candidates if c in df.columns), None)
if "oil_return" not in df.columns and oil_price_col is not None:
    df["oil_return"] = 100 * np.log(df[oil_price_col].astype(float)).diff()

crisis_cols = [
    "vietnam_devaluation_2011",
    "covid_2020",
    "fed_hiking_2022_2023",
]
for c in crisis_cols:
    if c not in df.columns:
        df[c] = 0

if "crisis_dummy" not in df.columns:
    df["crisis_dummy"] = df[crisis_cols].max(axis=1)

df_model = df.replace([np.inf, -np.inf], np.nan).copy()
df_model = df_model.dropna(subset=["fx_return"]).reset_index(drop=True)

global_shocks = [c for c in ["vix_change", "us10y_change", "dxy_return", "oil_return"] if c in df_model.columns]
domestic_controls = [c for c in ["vnindex_return"] if c in df_model.columns]
shock_base = global_shocks + domestic_controls
    
print("Model shape:", df_model.shape)
print("Global shocks:", global_shocks)
print("Domestic controls:", domestic_controls)

df_model.to_csv(OUT_DIR / "model_data_checked.csv", index=False)
display(df_model.head())

Raw shape: (4006, 25)
Date range: 2010-01-26 00:00:00 → 2025-12-31 00:00:00
Columns: ['Date', 'usd_vnd', 'log_usd_vnd', 'fx_return', 'abs_return', 'squared_return', 'rolling_vol_22d', 'vix', 'vix_change', 'us_10y', 'us10y_change', 'dxy', 'dxy_return', 'vnindex', 'vnindex_return', 'wti_oil', 'log_oil', 'oil_return', 'abs_oil_return', 'squared_oil_return', 'vietnam_fx_devaluation_2010', 'vietnam_fx_devaluation_2011', 'covid_2020', 'fed_hiking_2022_2023', 'crisis_dummy']
Model shape: (4006, 26)
Global shocks: ['vix_change', 'us10y_change', 'dxy_return', 'oil_return']
Domestic controls: ['vnindex_return']


,Date,usd_vnd,log_usd_vnd,fx_return,abs_return,squared_return,rolling_vol_22d,vix,vix_change,us_10y,us10y_change,dxy,dxy_return,vnindex,vnindex_return,wti_oil,log_oil,oil_return,abs_oil_return,squared_oil_return,vietnam_fx_devaluation_2010,vietnam_fx_devaluation_2011,covid_2020,fed_hiking_2022_2023,crisis_dummy,vietnam_devaluation_2011
0,2010-01-26,"18,469.000000",9.823849,0.000000,0.000000,0.000000,0.020919,24.550000,-0.860000,3.650000,-0.010000,78.430000,0.306472,497.900000,3.471911,74.670000,4.313078,-0.307549,0.307549,0.094586,0,0,0,0,0,0
1,2010-01-27,"18,474.043968",9.824122,0.027307,0.027307,0.000746,0.021482,23.140000,-1.410000,3.660000,0.010000,78.680000,0.318249,486.410000,-2.334736,73.640000,4.299188,-1.389005,1.389005,1.929335,0,0,0,0,0,0
2,2010-01-28,"18,469.608260",9.823882,-0.024013,0.024013,0.000577,0.021600,23.730000,0.590000,3.680000,0.020000,78.900002,0.279225,481.760000,-0.960582,73.620000,4.298917,-0.027163,0.027163,0.000738,0,0,0,0,0,0
3,2010-01-29,"18,474.043968",9.824122,0.024013,0.024013,0.000577,0.021795,24.620000,0.890000,3.630000,-0.050000,79.459999,0.707249,481.960000,0.041506,72.850000,4.288403,-1.051420,1.051420,1.105483,0,0,0,0,0,0
4,2010-02-01,"18,469.000000",9.823849,-0.027307,0.027307,0.000746,0.022088,22.590000,-2.030000,3.680000,0.050000,79.239998,-0.277254,486.950000,1.030033,74.410000,4.309590,2.118781,2.118781,4.489232,0,0,0,0,0,0


## 2. ARMA Mean Equation Selection

This section selects the ARMA baseline for the return mean equation only.
It is not mixed with the GARCH-family volatility comparison.

Important: `arch_model` supports AR terms in the mean equation but not MA terms. Therefore, the GARCH-family models below use AR(1), not the selected ARMA(p,q). This is not an error as long as it is stated clearly in the report.

In [9]:
# ============================================================
# 2. ARMA MODEL SELECTION FOR MEAN EQUATION ONLY
# ============================================================

y = df_model["fx_return"].astype(float).replace([np.inf, -np.inf], np.nan).dropna()

MAX_ARMA_P = 5
MAX_ARMA_Q = 5
MIN_LB_PVALUE = 0.05
LB_LAGS = [10, 20]
ARMA_SELECTION_CRITERION = "BIC"


def get_ljungbox_pvalues_from_residuals(residuals, lags=[10, 20], model_df=0, burn=0):
    resid = pd.Series(residuals).replace([np.inf, -np.inf], np.nan).dropna()

    if burn > 0 and len(resid) > burn:
        resid = resid.iloc[burn:]

    max_lag = max(lags)
    if len(resid) <= max_lag + model_df + 5:
        return {f"LB_p_lag{lag}": np.nan for lag in lags}

    try:
        lb = acorr_ljungbox(
            resid,
            lags=lags,
            return_df=True,
            model_df=model_df
        )

        return {
            f"LB_p_lag{lag}": float(lb.loc[lag, "lb_pvalue"])
            for lag in lags
        }

    except Exception:
        return {f"LB_p_lag{lag}": np.nan for lag in lags}


def get_arch_lm_pvalue(residuals, nlags=10, burn=0):
    resid = pd.Series(residuals).replace([np.inf, -np.inf], np.nan).dropna()

    if burn > 0 and len(resid) > burn:
        resid = resid.iloc[burn:]

    if len(resid) <= nlags + 5:
        return np.nan

    try:
        return float(het_arch(resid, nlags=nlags)[1])
    except Exception:
        return np.nan


def fit_arma_grid(y, max_p=5, max_q=5, show_progress=True):
    y = pd.Series(y).dropna().astype(float)

    rows = []
    fitted = {}

    candidate_orders = [
        (p, q)
        for p in range(max_p + 1)
        for q in range(max_q + 1)
        if not (p == 0 and q == 0)
    ]

    iterator = candidate_orders
    if tqdm is not None and show_progress:
        iterator = tqdm(candidate_orders, desc="Fitting ARMA grid", unit="model")

    for p, q in iterator:
        model_name = f"ARMA({p},{q})"

        try:
            model = SARIMAX(
                y,
                order=(p, 0, q),
                trend="c",
                enforce_stationarity=True,
                enforce_invertibility=True,
            )

            res = model.fit(disp=False, maxiter=1000)

            mle_retvals = getattr(res, "mle_retvals", {})
            converged = bool(mle_retvals.get("converged", False))
            message = str(mle_retvals.get("message", "OK"))

            burn = max(p, q)

            # Raw Ljung-Box: dùng để kiểm tra residual autocorrelation thực tế
            lb_raw = get_ljungbox_pvalues_from_residuals(
                res.resid,
                lags=LB_LAGS,
                model_df=0,
                burn=burn
            )

            # Adjusted Ljung-Box: tham khảo thêm, có trừ bậc tự do p + q
            lb_adj = get_ljungbox_pvalues_from_residuals(
                res.resid,
                lags=LB_LAGS,
                model_df=p + q,
                burn=burn
            )

            arch_p = get_arch_lm_pvalue(res.resid, nlags=10, burn=burn)

            lb10 = lb_raw["LB_p_lag10"]
            lb20 = lb_raw["LB_p_lag20"]

            diagnostic_pass = (
                np.isfinite(lb10)
                and np.isfinite(lb20)
                and lb10 >= MIN_LB_PVALUE
                and lb20 >= MIN_LB_PVALUE
            )

            rows.append({
                "model": model_name,
                "p": p,
                "q": q,
                "loglik": float(res.llf),
                "AIC": float(res.aic),
                "BIC": float(res.bic),
                "HQIC": float(res.hqic),
                "num_params": int(len(res.params)),
                "converged": converged,

                "LB_resid_p_lag10": lb_raw["LB_p_lag10"],
                "LB_resid_p_lag20": lb_raw["LB_p_lag20"],

                "LB_resid_p_lag10_adj_df": lb_adj["LB_p_lag10"],
                "LB_resid_p_lag20_adj_df": lb_adj["LB_p_lag20"],

                "diagnostic_pass": diagnostic_pass,
                "ARCH_LM_p_lag10": arch_p,

                "message": message,
                "error": "",
            })

            if converged:
                fitted[model_name] = res

        except Exception as e:
            rows.append({
                "model": model_name,
                "p": p,
                "q": q,
                "loglik": np.nan,
                "AIC": np.nan,
                "BIC": np.nan,
                "HQIC": np.nan,
                "num_params": np.nan,
                "converged": False,

                "LB_resid_p_lag10": np.nan,
                "LB_resid_p_lag20": np.nan,

                "LB_resid_p_lag10_adj_df": np.nan,
                "LB_resid_p_lag20_adj_df": np.nan,

                "diagnostic_pass": False,
                "ARCH_LM_p_lag10": np.nan,

                "message": "",
                "error": str(e)[:200],
            })

    table = pd.DataFrame(rows)

    table["AIC_rank"] = table["AIC"].rank(method="min")
    table["BIC_rank"] = table["BIC"].rank(method="min")
    table["HQIC_rank"] = table["HQIC"].rank(method="min")

    table = table.sort_values(
        ["converged", "diagnostic_pass", ARMA_SELECTION_CRITERION, "AIC"],
        ascending=[False, False, True, True],
        na_position="last"
    ).reset_index(drop=True)

    return table, fitted


if RUN_ARMA_GRID:
    arma_table, arma_fitted = fit_arma_grid(
        y,
        max_p=MAX_ARMA_P,
        max_q=MAX_ARMA_Q,
        show_progress=True
    )
else:
    arma_table, arma_fitted = fit_arma_grid(
        y,
        max_p=5,
        max_q=5,
        show_progress=False
    )


valid_arma = arma_table[
    (arma_table["converged"] == True)
    & np.isfinite(arma_table[ARMA_SELECTION_CRITERION])
].copy()

if valid_arma.empty:
    raise ValueError("No valid ARMA model found.")


clean_arma = valid_arma[
    (valid_arma["diagnostic_pass"] == True)
].copy()

if clean_arma.empty:
    display(valid_arma.sort_values(ARMA_SELECTION_CRITERION).head(15))

    raise ValueError(
        "No ARMA model passed Ljung-Box residual diagnostics at lag 10 and lag 20. "
        "Do not continue to GARCH. Increase MAX_ARMA_P/MAX_ARMA_Q or revise the mean equation."
    )


best_arma_row = clean_arma.sort_values(
    [ARMA_SELECTION_CRITERION, "AIC", "p", "q"],
    ascending=[True, True, True, True]
).iloc[0]

best_arma_name = best_arma_row["model"]
best_arma = arma_fitted[best_arma_name]

arma_table.to_csv(OUT_DIR / "table_1_arma_mean_selection.csv", index=False)

print("Best ARMA mean baseline among clean residual models:", best_arma_name)

display(
    arma_table[
        [
            "model", "p", "q",
            "loglik", "AIC", "BIC", "HQIC",
            "AIC_rank", "BIC_rank",
            "converged",
            "LB_resid_p_lag10",
            "LB_resid_p_lag20",
            "LB_resid_p_lag10_adj_df",
            "LB_resid_p_lag20_adj_df",
            "diagnostic_pass",
            "ARCH_LM_p_lag10",
            "error"
        ]
    ].head(20)
)

Fitting ARMA grid: 100%|██████████| 35/35 [01:00<00:00,  1.73s/model]

Best ARMA mean baseline among clean residual models: ARMA(3,2)


,model,p,q,loglik,AIC,BIC,HQIC,AIC_rank,BIC_rank,converged,LB_resid_p_lag10,LB_resid_p_lag20,LB_resid_p_lag10_adj_df,LB_resid_p_lag20_adj_df,diagnostic_pass,ARCH_LM_p_lag10,error
0,"ARMA(3,2)",3,2,580.307487,"-1,146.614973","-1,102.546134","-1,130.994907",1.000000,1.000000,True,0.325318,0.135941,0.043561,0.028933,True,0.000039,
1,"ARMA(2,3)",2,3,579.319667,"-1,144.639335","-1,100.570495","-1,129.019268",8.000000,2.000000,True,0.266461,0.101199,0.031090,0.019434,True,0.000038,
2,"ARMA(2,4)",2,4,580.691803,"-1,145.383607","-1,095.019218","-1,127.532102",4.000000,3.000000,True,0.366212,0.159217,0.027837,0.024391,True,0.000051,
3,"ARMA(2,5)",2,5,581.471344,"-1,144.942688","-1,088.282752","-1,124.859745",5.000000,4.000000,True,0.439057,0.177861,0.018427,0.018965,True,0.000038,
4,"ARMA(5,2)",5,2,581.470579,"-1,144.941158","-1,088.281221","-1,124.858215",6.000000,5.000000,True,0.349247,0.154073,0.011161,0.015177,True,0.000037,
5,"ARMA(4,3)",4,3,580.100326,"-1,142.200652","-1,085.540715","-1,122.117709",10.000000,6.000000,True,0.172082,0.079935,0.002878,0.005717,True,0.000042,
6,"ARMA(3,4)",3,4,579.350101,"-1,140.700202","-1,084.040265","-1,120.617259",13.000000,7.000000,True,0.386177,0.147188,0.013835,0.014149,True,0.000053,
7,"ARMA(3,5)",3,5,583.014277,"-1,146.028554","-1,083.073069","-1,123.714173",2.000000,8.000000,True,0.829848,0.353491,0.054374,0.040260,True,0.000039,
8,"ARMA(4,4)",4,4,583.011203,"-1,146.022406","-1,083.066921","-1,123.708025",3.000000,9.000000,True,0.832498,0.361146,0.055268,0.041932,True,0.000039,
9,"ARMA(5,3)",5,3,580.687681,"-1,141.375362","-1,078.419877","-1,119.060981",12.000000,11.000000,True,0.643014,0.303056,0.019694,0.030255,True,0.000049,


In [10]:
arma_resid = pd.Series(best_arma.resid).replace([np.inf, -np.inf], np.nan).dropna()

resid_diag = pd.DataFrame([{
    "selected_arma_model": best_arma_name,
    "n_obs": len(arma_resid),
    "resid_mean": arma_resid.mean(),
    "resid_std": arma_resid.std(ddof=1),
    "LB_resid_p_lag10": float(acorr_ljungbox(arma_resid, lags=[10], return_df=True)["lb_pvalue"].iloc[0]),
    "LB_resid_p_lag20": float(acorr_ljungbox(arma_resid, lags=[20], return_df=True)["lb_pvalue"].iloc[0]),
    "LB_sq_resid_p_lag10": float(acorr_ljungbox(arma_resid ** 2, lags=[10], return_df=True)["lb_pvalue"].iloc[0]),
    "LB_sq_resid_p_lag20": float(acorr_ljungbox(arma_resid ** 2, lags=[20], return_df=True)["lb_pvalue"].iloc[0]),
    "ARCH_LM_p_lag10": float(het_arch(arma_resid, nlags=10)[1]),
}])

display(resid_diag)


,selected_arma_model,n_obs,resid_mean,resid_std,LB_resid_p_lag10,LB_resid_p_lag20,LB_sq_resid_p_lag10,LB_sq_resid_p_lag20,ARCH_LM_p_lag10
0,"ARMA(3,2)",4006,0.000027,0.209425,0.332095,0.131550,0.000001,0.000129,0.000048


**ARMA(3,2) là mean equation phù hợp: residual không còn tự tương quan, nhưng vẫn còn ARCH effect rõ rệt. Vì vậy tiếp tục chạy GARCH/EGARCH/GJR-GARCH cho volatility equation là hợp lí.**

## 3. GARCH-family Volatility Models

These models are compared only within the GARCH-family block. Broken numerical results are filtered out using sanity checks, not only by optimizer convergence flags.

In [11]:
# ============================================================
# 3. GARCH-FAMILY VOLATILITY MODEL COMPARISON
#    Using ARMA(3,2) residuals + arch_model(mean="Zero")
# ============================================================

from arch import arch_model
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

# ------------------------------------------------------------
# 3.1. Get ARMA(3,2) residuals
# ---------------------
arma_res = best_arma
arma_name = best_arma_name
eps = pd.Series(arma_res.resid).replace([np.inf, -np.inf], np.nan).dropna()

# Bỏ vài quan sát đầu do ARMA initialization
ARMA_BURN_IN = max(int(best_arma_row["p"]), int(best_arma_row["q"])) if "best_arma_row" in globals() else 3
eps = eps.iloc[ARMA_BURN_IN:]

print(f"Using residuals from mean equation: {arma_name}")
print(f"Residual sample size for GARCH-family models: {len(eps)}")


# ------------------------------------------------------------
# 3.2. GARCH-family model specifications
# ------------------------------------------------------------

GARCH_SPECS = [
    (
        "ARMA(3,2)-GARCH(1,1)-t",
        {
            "mean": "Zero",
            "vol": "GARCH",
            "p": 1,
            "o": 0,
            "q": 1,
            "dist": "t",
        },
    ),
    (
        "ARMA(3,2)-EGARCH(1,1)-t",
        {
            "mean": "Zero",
            "vol": "EGARCH",
            "p": 1,
            "o": 0,
            "q": 1,
            "dist": "t",
        },
    ),
    (
        "ARMA(3,2)-EGARCH(1,1)-asym-t",
        {
            "mean": "Zero",
            "vol": "EGARCH",
            "p": 1,
            "o": 1,
            "q": 1,
            "dist": "t",
        },
    ),
    (
        "ARMA(3,2)-GJR-GARCH(1,1)-t",
        {
            "mean": "Zero",
            "vol": "GARCH",
            "p": 1,
            "o": 1,
            "q": 1,
            "dist": "t",
        },
    ),
]


# ------------------------------------------------------------
# 3.3. Safe diagnostic helpers
# ------------------------------------------------------------

def safe_ljung_box(x, lag):
    try:
        x = pd.Series(x).replace([np.inf, -np.inf], np.nan).dropna()
        if len(x) <= lag + 5:
            return np.nan
        return float(acorr_ljungbox(x, lags=[lag], return_df=True)["lb_pvalue"].iloc[0])
    except Exception:
        return np.nan


def safe_arch_lm(x, lag=10):
    try:
        x = pd.Series(x).replace([np.inf, -np.inf], np.nan).dropna()
        if len(x) <= lag + 5:
            return np.nan
        return float(het_arch(x, nlags=lag)[1])
    except Exception:
        return np.nan


def fit_arch_model_safe(eps, name, kwargs):
    try:
        am = arch_model(
            eps,
            rescale=False,
            **kwargs
        )

        res = am.fit(
            disp="off",
            update_freq=0,
            show_warning=False,
            options={"maxiter": 5000}
        )

        return res, None

    except Exception as e:
        return None, str(e)[:300]


def garch_diagnostics(res, name):
    z = pd.Series(res.std_resid).replace([np.inf, -np.inf], np.nan).dropna()
    z2 = z ** 2
    n = len(z)

    lb_resid_10 = safe_ljung_box(z, 10)
    lb_resid_20 = safe_ljung_box(z, 20)
    lb_sq_10 = safe_ljung_box(z2, 10)
    lb_sq_20 = safe_ljung_box(z2, 20)
    arch_10 = safe_arch_lm(z, 10)

    out = {
        "model": name,
        "n_obs": n,
        "converged": bool(res.convergence_flag == 0),
        "convergence_flag": int(res.convergence_flag),
        "loglik": float(res.loglikelihood),
        "AIC": float(res.aic),
        "BIC": float(res.bic),
        "num_params": int(res.num_params),

        "std_resid_mean": float(z.mean()) if n else np.nan,
        "std_resid_std": float(z.std(ddof=1)) if n else np.nan,
        "std_resid_min": float(z.min()) if n else np.nan,
        "std_resid_max": float(z.max()) if n else np.nan,
        "abs_std_resid_gt_3": int((z.abs() > 3).sum()) if n else np.nan,
        "abs_std_resid_gt_3_ratio": float((z.abs() > 3).mean()) if n else np.nan,

        "LB_resid_p_lag10": lb_resid_10,
        "LB_resid_p_lag20": lb_resid_20,
        "LB_sq_resid_p_lag10": lb_sq_10,
        "LB_sq_resid_p_lag20": lb_sq_20,
        "ARCH_LM_p_lag10": arch_10,
    }

    finite_core = np.isfinite([
        out["loglik"],
        out["AIC"],
        out["BIC"],
        out["std_resid_std"],
    ]).all()

    sane_std = (
        0.60 <= out["std_resid_std"] <= 1.40
        if np.isfinite(out["std_resid_std"])
        else False
    )

    sane_tail = (
        out["abs_std_resid_gt_3_ratio"] <= 0.05
        if np.isfinite(out["abs_std_resid_gt_3_ratio"])
        else False
    )

    max_abs_z = max(
        abs(out["std_resid_min"]),
        abs(out["std_resid_max"])
    ) if np.isfinite(out["std_resid_min"]) and np.isfinite(out["std_resid_max"]) else np.inf

    sane_extreme = max_abs_z <= 25

    out["passes_sanity"] = bool(
        out["converged"]
        and finite_core
        and sane_std
        and sane_tail
        and sane_extreme
    )

    # Diagnostic pass chính cho volatility model:
    # standardized residual squared không còn ARCH/volatility clustering rõ rệt.
    out["passes_volatility_diagnostics"] = bool(
        np.isfinite(lb_sq_10)
        and np.isfinite(lb_sq_20)
        and np.isfinite(arch_10)
        and lb_sq_10 > 0.05
        and lb_sq_20 > 0.05
        and arch_10 > 0.05
    )

    # Diagnostic pass đầy đủ:
    # residual và squared residual đều sạch.
    out["passes_full_diagnostics"] = bool(
        np.isfinite(lb_resid_10)
        and np.isfinite(lb_resid_20)
        and np.isfinite(lb_sq_10)
        and np.isfinite(lb_sq_20)
        and np.isfinite(arch_10)
        and lb_resid_10 > 0.05
        and lb_resid_20 > 0.05
        and lb_sq_10 > 0.05
        and lb_sq_20 > 0.05
        and arch_10 > 0.05
    )

    return out


# ------------------------------------------------------------
# 3.4. Fit GARCH-family models
# ------------------------------------------------------------

garch_fits = {}
garch_diag_rows = []

for name, kwargs in GARCH_SPECS:
    res, err = fit_arch_model_safe(eps, name, kwargs)

    if res is None:
        garch_diag_rows.append({
            "model": name,
            "n_obs": np.nan,
            "converged": False,
            "convergence_flag": np.nan,
            "loglik": np.nan,
            "AIC": np.nan,
            "BIC": np.nan,
            "num_params": np.nan,
            "passes_sanity": False,
            "passes_volatility_diagnostics": False,
            "passes_full_diagnostics": False,
            "error": err,
        })

    else:
        garch_fits[name] = res
        row = garch_diagnostics(res, name)
        row["error"] = ""
        garch_diag_rows.append(row)


garch_table = pd.DataFrame(garch_diag_rows)

garch_table = garch_table.sort_values(
    [
        "passes_sanity",
        "passes_volatility_diagnostics",
        "passes_full_diagnostics",
        "BIC",
        "AIC",
    ],
    ascending=[False, False, False, True, True],
    na_position="last",
).reset_index(drop=True)


# ------------------------------------------------------------
# 3.5. Select best standard GARCH-family model
# ------------------------------------------------------------

valid_garch = garch_table[
    (garch_table["passes_sanity"] == True)
    & (garch_table["passes_volatility_diagnostics"] == True)
    & np.isfinite(garch_table["BIC"])
].copy()

if valid_garch.empty:
    print(
        "Warning: No GARCH-family model fully passed volatility diagnostics. "
        "Selecting among sane converged models by BIC."
    )

    valid_garch = garch_table[
        (garch_table["passes_sanity"] == True)
        & np.isfinite(garch_table["BIC"])
    ].copy()

if valid_garch.empty:
    raise ValueError(
        "No GARCH-family model passed sanity checks. "
        "Inspect table_2_garch_family_comparison_and_diagnostics.csv."
    )


best_garch_row = valid_garch.sort_values(
    ["BIC", "AIC"],
    ascending=[True, True]
).iloc[0]

best_garch_name = best_garch_row["model"]
best_garch = garch_fits[best_garch_name]


# ------------------------------------------------------------
# 3.6. Save and display results
# ------------------------------------------------------------

garch_table.to_csv(
    OUT_DIR / "table_2_garch_family_comparison_and_diagnostics.csv",
    index=False
)

print("Best standard GARCH-family model:", best_garch_name)

display(garch_table)

print(best_garch.summary())

Using residuals from mean equation: ARMA(3,2)
Residual sample size for GARCH-family models: 4003
Best standard GARCH-family model: ARMA(3,2)-EGARCH(1,1)-t


,model,n_obs,converged,convergence_flag,loglik,AIC,BIC,num_params,std_resid_mean,std_resid_std,std_resid_min,std_resid_max,abs_std_resid_gt_3,abs_std_resid_gt_3_ratio,LB_resid_p_lag10,LB_resid_p_lag20,LB_sq_resid_p_lag10,LB_sq_resid_p_lag20,ARCH_LM_p_lag10,passes_sanity,passes_volatility_diagnostics,passes_full_diagnostics,error
0,"ARMA(3,2)-EGARCH(1,1)-t",4003,True,0,"3,346.386910","-6,684.773820","-6,659.594623",4,-0.051892,0.959946,-8.010221,16.472207,47,0.011741,0.000062,0.000085,0.996283,0.999981,0.996230,True,True,False,
1,"ARMA(3,2)-EGARCH(1,1)-asym-t",4003,True,0,"3,346.661738","-6,683.323475","-6,651.849479",5,-0.052227,0.959415,-7.955089,16.522996,49,0.012241,0.000060,0.000080,0.996104,0.999982,0.996115,True,True,False,
2,"ARMA(3,2)-GJR-GARCH(1,1)-t",4003,True,0,"3,146.659452","-6,283.318903","-6,251.844906",5,-0.040606,1.059723,-8.813972,17.542702,65,0.016238,0.000011,0.000126,0.000000,0.000000,0.000000,True,False,False,
3,"ARMA(3,2)-GARCH(1,1)-t",4003,True,0,"3,140.888696","-6,273.777391","-6,248.598194",4,-0.040098,1.054525,-8.701384,17.536240,64,0.015988,0.000011,0.000131,0.000000,0.000000,0.000000,True,False,False,


                          Zero Mean - EGARCH Model Results                          
Dep. Variable:                         None   R-squared:                       0.000
Mean Model:                       Zero Mean   Adj. R-squared:                  0.000
Vol Model:                           EGARCH   Log-Likelihood:                3346.39
Distribution:      Standardized Student's t   AIC:                          -6684.77
Method:                  Maximum Likelihood   BIC:                          -6659.59
                                              No. Observations:                 4003
Date:                      Fri, May 29 2026   Df Residuals:                     4003
Time:                              00:37:45   Df Model:                            0
                             Volatility Model                             
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
omega     

In [12]:
# ============================================================
# CREATE CONDITIONAL VOLATILITY SERIES FROM BEST STANDARD GARCH
# ============================================================

# Conditional volatility từ best standard GARCH-family model
cond_vol = pd.Series(
    best_garch.conditional_volatility,
    index=eps.index,
    name="cond_vol"
)

cond_var = cond_vol ** 2
log_cond_vol = np.log(cond_vol.replace(0, np.nan))
log_cond_var = np.log(cond_var.replace(0, np.nan))

# Gắn lại vào df_model theo index
df_model["selected_cond_vol"] = np.nan
df_model["selected_cond_var"] = np.nan
df_model["log_selected_cond_vol"] = np.nan
df_model["log_selected_cond_var"] = np.nan

df_model.loc[cond_vol.index, "selected_cond_vol"] = cond_vol
df_model.loc[cond_var.index, "selected_cond_var"] = cond_var
df_model.loc[log_cond_vol.index, "log_selected_cond_vol"] = log_cond_vol
df_model.loc[log_cond_var.index, "log_selected_cond_var"] = log_cond_var

# Nếu các cell sau vẫn đang gọi tên cũ log_egarch_vol, tạo alias để tránh lỗi
df_model["log_egarch_vol"] = df_model["log_selected_cond_vol"]

print("Created volatility columns:")
print([
    "selected_cond_vol",
    "selected_cond_var",
    "log_selected_cond_vol",
    "log_selected_cond_var",
    "log_egarch_vol"
])

Created volatility columns:
['selected_cond_vol', 'selected_cond_var', 'log_selected_cond_vol', 'log_selected_cond_var', 'log_egarch_vol']


## 4. Extract Selected Standard GARCH-family Conditional Volatility

This is used for diagnostics and the supplementary HAC shock-volatility regressions.


In [13]:
# ============================================================
# 4. CONDITIONAL VOLATILITY FROM SELECTED STANDARD GARCH MODEL
#    Correct naming: selected_cond_vol, not egarch_cond_vol
# ============================================================

# Use the volatility model selected in Step 3
vol_source_name = best_garch_name
vol_source_res = garch_fits[vol_source_name]

# Conditional volatility from selected model
selected_cond_vol = pd.Series(
    np.asarray(vol_source_res.conditional_volatility),
    index=eps.index,
    name="selected_cond_vol"
)

selected_cond_var = selected_cond_vol ** 2

log_selected_cond_vol = np.log(
    selected_cond_vol.replace(0, np.nan)
)

log_selected_cond_var = np.log(
    selected_cond_var.replace(0, np.nan)
)

# ------------------------------------------------------------
# Attach selected volatility variables back to df_model
# ------------------------------------------------------------

df_model = df_model.copy()

# Remove old misleading EGARCH names if they already exist
old_wrong_cols = [
    "egarch_cond_vol",
    "log_egarch_vol"
]

df_model = df_model.drop(
    columns=[c for c in old_wrong_cols if c in df_model.columns],
    errors="ignore"
)

# Create correctly named columns
df_model["selected_cond_vol"] = np.nan
df_model["selected_cond_var"] = np.nan
df_model["log_selected_cond_vol"] = np.nan
df_model["log_selected_cond_var"] = np.nan
df_model["vol_source_model"] = vol_source_name

# Align by eps.index
df_model.loc[selected_cond_vol.index, "selected_cond_vol"] = selected_cond_vol
df_model.loc[selected_cond_var.index, "selected_cond_var"] = selected_cond_var
df_model.loc[log_selected_cond_vol.index, "log_selected_cond_vol"] = log_selected_cond_vol
df_model.loc[log_selected_cond_var.index, "log_selected_cond_var"] = log_selected_cond_var

# ------------------------------------------------------------
# Save output
# ------------------------------------------------------------

df_model.to_csv(
    OUT_DIR / "model_data_with_selected_conditional_volatility.csv",
    index=False
)

print("Volatility source model:", vol_source_name)
print("Saved conditional volatility variables:")
print([
    "selected_cond_vol",
    "selected_cond_var",
    "log_selected_cond_vol",
    "log_selected_cond_var",
    "vol_source_model"
])

# ------------------------------------------------------------
# Display preview
# ------------------------------------------------------------

preview_cols = []

if "Date" in df_model.columns:
    preview_cols.append("Date")
preview_cols += [
    "fx_return",
    "selected_cond_vol",
    "log_selected_cond_vol",
    "vol_source_model"
]

if "shock_base" in globals():
    preview_cols += [c for c in shock_base if c in df_model.columns]

preview_cols = [c for c in preview_cols if c in df_model.columns]

display(
    df_model[preview_cols]
    .dropna(subset=["selected_cond_vol"])
    .head()
)

Volatility source model: ARMA(3,2)-EGARCH(1,1)-t
Saved conditional volatility variables:
['selected_cond_vol', 'selected_cond_var', 'log_selected_cond_vol', 'log_selected_cond_var', 'vol_source_model']


,Date,fx_return,selected_cond_vol,log_selected_cond_vol,vol_source_model,vix_change,us10y_change,dxy_return,oil_return,vnindex_return
3,2010-01-29,0.024013,0.877354,-0.130845,"ARMA(3,2)-EGARCH(1,1)-t",0.890000,-0.050000,0.707249,-1.051420,0.041506
4,2010-02-01,-0.027307,0.696833,-0.361210,"ARMA(3,2)-EGARCH(1,1)-t",-2.030000,0.050000,-0.277254,2.118781,1.030033
5,2010-02-02,0.030078,0.564468,-0.571871,"ARMA(3,2)-EGARCH(1,1)-t",-1.110000,-0.010000,-0.290674,3.693864,0.201050
6,2010-02-03,-0.002771,0.459040,-0.778618,"ARMA(3,2)-EGARCH(1,1)-t",0.120000,0.060000,0.454604,-0.324318,1.535504
7,2010-02-04,-0.027307,0.375712,-0.978931,"ARMA(3,2)-EGARCH(1,1)-t",4.480000,-0.110000,0.690561,-5.104713,1.671192


## 5. Custom EGARCH-X

### **DETECT DEVALUATION DATE FROM ARMA RESIDUALS ONLY**

In [14]:
# 1. Get ARMA residuals
if "eps" in globals():
    eps_detect = pd.Series(eps)
elif "best_arma" in globals():
    eps_detect = pd.Series(best_arma.resid)
    burn = max(int(best_arma_row["p"]), int(best_arma_row["q"])) if "best_arma_row" in globals() else 3
    eps_detect = eps_detect.iloc[burn:]
else:
    eps_detect = pd.Series(selected_arma_res.resid).iloc[3:]

eps_detect = eps_detect.replace([np.inf, -np.inf], np.nan).dropna().astype(float)

# 2. Align residuals with df_model
detect_df = df_model.copy()
detect_df["Date"] = pd.to_datetime(detect_df["Date"])

detect_df["arma_resid"] = np.nan
detect_df.loc[eps_detect.index, "arma_resid"] = eps_detect
detect_df = detect_df.dropna(subset=["arma_resid"]).copy()

# 3. Detect largest ARMA residual day
detect_df["abs_arma_resid"] = detect_df["arma_resid"].abs()

report_cols = [
    "Date",
    "arma_resid",
    "abs_arma_resid",
    "vix_change",
    "us10y_change",
    "dxy_return",
    "oil_return",
    "vnindex_return",
    "crisis_dummy",
]
report_cols = [c for c in report_cols if c in detect_df.columns]

extreme_arma_resid_table = (
    detect_df
    .sort_values("abs_arma_resid", ascending=False)
    [report_cols]
    .head(15)
    .reset_index(drop=True)
)

display(extreme_arma_resid_table)

# 4. Exploratory table only; do not define the policy dummy from the largest residual
largest_arma_resid_date = pd.to_datetime(extreme_arma_resid_table.loc[0, "Date"])

# Policy-event dummy is fixed from the known Vietnam devaluation event.
# The residual table above is used only to document extreme observations.
DEVALUATION_DATE = pd.Timestamp("2010-02-11")
DEVALUATION_DUMMY_COL = "devaluation_20100211_dummy"

df_model[DEVALUATION_DUMMY_COL] = (
    pd.to_datetime(df_model["Date"]).eq(DEVALUATION_DATE).astype(int)
)

print("Largest ARMA residual date:", largest_arma_resid_date.date())
print("Policy devaluation dummy date:", DEVALUATION_DATE.date())
print("Created dummy:", DEVALUATION_DUMMY_COL)
print("Dummy count:", int(df_model[DEVALUATION_DUMMY_COL].sum()))

,Date,arma_resid,abs_arma_resid,vix_change,us10y_change,dxy_return,oil_return,vnindex_return,crisis_dummy
0,2011-02-14,6.298831,6.298831,0.260000,-0.020000,0.191000,-0.868795,-1.146981,1
1,2010-02-18,-2.371792,2.371792,-1.090000,0.050000,0.024884,2.176225,0.000000,1
2,2010-03-11,2.188706,2.188706,-0.510000,0.000000,-0.161718,0.036547,-0.196133,1
3,2010-02-26,2.166829,2.166829,-0.600000,-0.030000,-0.533666,2.193988,0.387136,1
4,2010-02-24,2.163148,2.163148,-1.100000,0.010000,0.000000,1.439782,-0.343130,1
5,2010-03-04,-2.053352,2.053352,-0.110000,-0.020000,0.722557,-0.868923,0.900686,1
6,2010-02-11,2.009600,2.009600,-1.440000,0.010000,-0.049995,1.001945,1.822728,1
7,2010-02-25,-1.983457,1.983457,-0.170000,-0.060000,-0.074236,-2.231613,0.080842,1
8,2010-03-08,1.783811,1.783811,0.370000,0.030000,0.000000,0.428528,1.494455,1
9,2010-08-18,1.739807,1.739807,0.260000,0.000000,-0.012164,-0.489581,-1.747577,1


Largest ARMA residual date: 2011-02-14
Policy devaluation dummy date: 2010-02-11
Created dummy: devaluation_20100211_dummy
Dummy count: 1


The extreme ARMA-residual table is exploratory. The EGARCH-X dummy is not mechanically defined from the largest residual. It is fixed to the known Vietnam policy devaluation event on `2010-02-11`, treated as a policy-related exchange-rate volatility shock.

In [15]:
MAXITER = 3000
DIAGNOSTIC_BURN_IN = 60
COMPUTE_PVALUES = True

# Fixed policy-event date, not mechanically detected from the largest residual
DEVALUATION_DATE = pd.Timestamp("2010-02-11")
DEVALUATION_DUMMY_COL = "devaluation_20100211_dummy"

In [16]:
# ============================================================
# 5.1 PREPARE DATA
# ============================================================

def get_arma_residuals():
    if "eps" in globals():
        eps0 = pd.Series(eps)
    elif "best_arma" in globals():
        eps0 = pd.Series(best_arma.resid)
    else:
        eps0 = pd.Series(selected_arma_res.resid)

    eps0 = eps0.replace([np.inf, -np.inf], np.nan).dropna()

    if "best_arma_row" in globals():
        burn = max(int(best_arma_row["p"]), int(best_arma_row["q"]))
    else:
        burn = 3

    return eps0.iloc[burn:]


shock_df = df_model.copy()

if "Date" in shock_df.columns:
    shock_df["Date"] = pd.to_datetime(shock_df["Date"])
    shock_df = shock_df.sort_values("Date")

eps = get_arma_residuals()

shock_df["arma_resid"] = np.nan
shock_df.loc[eps.index, "arma_resid"] = eps

shock_df[DEVALUATION_DUMMY_COL] = (
    shock_df["Date"].eq(DEVALUATION_DATE).astype(int)
    if "Date" in shock_df.columns
    else 0
)

if "shock_base" not in globals():
    shock_base = ["vix_change", "us10y_change", "dxy_return", "oil_return", "vnindex_return"]

SHOCK_BASE = [c for c in shock_base if c in shock_df.columns]

for col in SHOCK_BASE:
    std = shock_df[col].std(ddof=0)

    if np.isfinite(std) and std > 0:
        shock_df[f"z_{col}"] = (shock_df[col] - shock_df[col].mean()) / std
        shock_df[f"abs_z_{col}"] = shock_df[f"z_{col}"].abs()
    else:
        shock_df[f"z_{col}"] = np.nan
        shock_df[f"abs_z_{col}"] = np.nan

    shock_df[f"z_{col}_lag1"] = shock_df[f"z_{col}"].shift(1)
    shock_df[f"abs_z_{col}_lag1"] = shock_df[f"abs_z_{col}"].shift(1)

shock_df = shock_df.drop(
    columns=[c for c in ["egarch_cond_vol", "log_egarch_vol"] if c in shock_df.columns],
    errors="ignore"
)

print("Residual sample size:", len(eps))
print("Shock variables:", SHOCK_BASE)
print("Devaluation dummy count:", int(shock_df[DEVALUATION_DUMMY_COL].sum()))


Residual sample size: 4000
Shock variables: ['vix_change', 'us10y_change', 'dxy_return', 'oil_return', 'vnindex_return']
Devaluation dummy count: 1


In [17]:
# ============================================================
# 5.2 HELPERS: DIAGNOSTICS, LIKELIHOOD, STARTING VALUES
# ============================================================

def clean_series(x):
    return pd.Series(x).replace([np.inf, -np.inf], np.nan).dropna()


def safe_ljung_box(x, lag):
    x = clean_series(x)
    if len(x) <= lag + 5:
        return np.nan

    try:
        return float(acorr_ljungbox(x, lags=[lag], return_df=True)["lb_pvalue"].iloc[0])
    except Exception:
        return np.nan


def safe_arch_lm(x, lag=10):
    x = clean_series(x)
    if len(x) <= lag + 5:
        return np.nan

    try:
        return float(het_arch(x, nlags=lag)[1])
    except Exception:
        return np.nan


def expected_abs_std_t(nu):
    return (
        2.0
        * np.sqrt(nu - 2.0)
        * np.exp(gammaln((nu + 1.0) / 2.0) - gammaln(nu / 2.0))
        / ((nu - 1.0) * np.sqrt(np.pi))
    )


def unpack_params(theta, k):
    omega = theta[0]
    alpha = theta[1]
    gamma = theta[2]
    beta = np.tanh(theta[3])
    nu = 2.05 + 97.95 * expit(theta[4])
    delta = np.asarray(theta[5:5 + k]) if k else np.array([])

    return omega, alpha, gamma, beta, nu, delta


def egarchx_filter(theta, eps_arr, X_arr):
    eps_arr = np.asarray(eps_arr, dtype=float)
    X_arr = np.asarray(X_arr, dtype=float)

    n = len(eps_arr)
    k = X_arr.shape[1] if X_arr.ndim == 2 else 0
    omega, alpha, gamma, beta, nu, delta = unpack_params(theta, k)

    h = np.zeros(n)
    logh = np.zeros(n)
    z = np.zeros(n)

    h[0] = max(np.nanvar(eps_arr), 1e-4)
    logh[0] = np.log(h[0])
    z[0] = eps_arr[0] / np.sqrt(h[0])

    eabs = expected_abs_std_t(nu)

    for t in range(1, n):
        x_effect = float(X_arr[t] @ delta) if k else 0.0

        logh[t] = (
            omega
            + beta * logh[t - 1]
            + alpha * (abs(z[t - 1]) - eabs)
            + gamma * z[t - 1]
            + x_effect
        )

        logh[t] = float(np.clip(logh[t], -25.0, 10.0))
        h[t] = np.exp(logh[t])
        z[t] = eps_arr[t] / np.sqrt(h[t])

        if not np.isfinite(z[t]) or abs(z[t]) > 1e6:
            raise FloatingPointError("Invalid EGARCH-X recursion.")

    return h, logh, z


def neg_loglik(theta, eps_arr, X_arr):
    try:
        h, _, _ = egarchx_filter(theta, eps_arr, X_arr)
        k = X_arr.shape[1] if X_arr.ndim == 2 else 0
        _, _, _, _, nu, _ = unpack_params(theta, k)

        eps_i = eps_arr[1:]
        h_i = h[1:]

        if np.any(~np.isfinite(h_i)) or np.any(h_i <= 0):
            return 1e12

        ll = (
            gammaln((nu + 1.0) / 2.0)
            - gammaln(nu / 2.0)
            - 0.5 * np.log(np.pi * (nu - 2.0))
            - 0.5 * np.log(h_i)
            - ((nu + 1.0) / 2.0)
            * np.log1p((eps_i ** 2) / ((nu - 2.0) * h_i))
        )

        return -float(np.sum(ll)) if np.all(np.isfinite(ll)) else 1e12

    except Exception:
        return 1e12


def make_starting_values(eps_series, k):
    eps_series = pd.Series(eps_series).dropna().astype(float)

    try:
        res = arch_model(
            eps_series,
            mean="Zero",
            vol="EGARCH",
            p=1,
            o=1,
            q=1,
            dist="t",
            rescale=False
        ).fit(
            disp="off",
            update_freq=0,
            show_warning=False,
            options={"maxiter": 5000}
        )

        p = res.params
        omega0 = float(p.get("omega", np.log(eps_series.var()) * 0.05))
        alpha0 = float(p.get("alpha[1]", 0.10))
        gamma0 = float(p.get("gamma[1]", 0.00))
        beta0 = float(p.get("beta[1]", 0.90))
        nu0 = float(p.get("nu", 8.0))

    except Exception:
        omega0 = float(np.log(eps_series.var())) * 0.05 if eps_series.var() > 0 else -0.1
        alpha0, gamma0, beta0, nu0 = 0.10, 0.00, 0.90, 8.0

    beta0 = float(np.clip(beta0, -0.98, 0.98))
    nu0 = float(np.clip(nu0, 2.10, 99.0))
    nu_scaled = np.clip((nu0 - 2.05) / 97.95, 1e-5, 1 - 1e-5)

    return np.r_[
        omega0,
        alpha0,
        gamma0,
        np.arctanh(beta0),
        logit(nu_scaled),
        np.zeros(k)
    ]


In [18]:
# ============================================================
# 5.3 APPROXIMATE P-VALUES
# ============================================================

def numerical_hessian(func, theta, args=(), step=1e-4):
    theta = np.asarray(theta, dtype=float)
    n = len(theta)
    h = step * np.maximum(1.0, np.abs(theta))

    H = np.zeros((n, n))

    for i in range(n):
        ei = np.zeros(n)
        ei[i] = h[i]

        for j in range(i, n):
            ej = np.zeros(n)
            ej[j] = h[j]

            fpp = func(theta + ei + ej, *args)
            fpm = func(theta + ei - ej, *args)
            fmp = func(theta - ei + ej, *args)
            fmm = func(theta - ei - ej, *args)

            value = (fpp - fpm - fmp + fmm) / (4.0 * h[i] * h[j])
            H[i, j] = value
            H[j, i] = value

    return H


def coefficient_table(model_name, theta, x_cols, eps_arr, X_arr, bounds, compute_pvalues=True):
    k = len(x_cols)
    omega, alpha, gamma, beta, nu, delta = unpack_params(theta, k)

    names = (
        ["omega", "alpha_abs_shock", "gamma_asymmetry", "beta_log_variance", "nu_student_t"]
        + [f"delta_{col}" for col in x_cols]
    )

    coefs = np.r_[omega, alpha, gamma, beta, nu, delta]

    raw_to_coef_grad = np.ones(len(theta))
    raw_to_coef_grad[3] = 1.0 - beta ** 2

    s_nu = expit(theta[4])
    raw_to_coef_grad[4] = 97.95 * s_nu * (1.0 - s_nu)

    at_bound = np.array([
        np.isclose(theta[i], bounds[i][0], atol=1e-4)
        or np.isclose(theta[i], bounds[i][1], atol=1e-4)
        for i in range(len(theta))
    ])

    se = np.full(len(theta), np.nan)

    if compute_pvalues:
        try:
            H = numerical_hessian(neg_loglik, theta, args=(eps_arr, X_arr))
            H = 0.5 * (H + H.T)

            cov_raw = np.linalg.pinv(H)
            var_raw = np.diag(cov_raw)

            valid = np.isfinite(var_raw) & (var_raw > 0)
            se[valid] = np.sqrt(var_raw[valid]) * np.abs(raw_to_coef_grad[valid])

        except Exception:
            pass

    t_stat = coefs / se
    p_value = 2.0 * norm.sf(np.abs(t_stat))

    return pd.DataFrame({
        "model": model_name,
        "parameter": names,
        "coef": coefs,
        "std_error": se,
        "t_stat": t_stat,
        "p_value": p_value,
        "significant_10pct": p_value < 0.10,
        "significant_5pct": p_value < 0.05,
        "significant_1pct": p_value < 0.01,
        "at_bound": at_bound,
    })


In [19]:
# ============================================================
# 5.4 FIT ONE MODEL
# ============================================================

def fit_egarchx(data, y_col, x_cols, model_name, maxiter=MAXITER):
    base_cols = ["Date", y_col] if "Date" in data.columns else [y_col]
    report_cols = [c for c in SHOCK_BASE + ["crisis_dummy", DEVALUATION_DUMMY_COL] if c in data.columns]
    cols = list(dict.fromkeys(base_cols + x_cols + report_cols))

    d = data[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if "Date" in d.columns:
        d = d.sort_values("Date")

    eps_arr = d[y_col].astype(float).to_numpy()
    X_arr = d[x_cols].astype(float).to_numpy() if x_cols else np.empty((len(d), 0))

    k = X_arr.shape[1]
    theta0 = make_starting_values(d[y_col], k)

    bounds = [(-10, 10), (-5, 5), (-5, 5), (-4, 4), (-8, 8)] + [(-3, 3)] * k

    opt = minimize(
        neg_loglik,
        theta0,
        args=(eps_arr, X_arr),
        method="L-BFGS-B",
        bounds=bounds,
        options={"maxiter": maxiter, "ftol": 1e-8, "gtol": 1e-5, "maxls": 50},
    )

    theta = opt.x
    loglik = -neg_loglik(theta, eps_arr, X_arr)
    nobs = len(eps_arr) - 1
    npar = len(theta)

    h, logh, z = egarchx_filter(theta, eps_arr, X_arr)

    burn = min(DIAGNOSTIC_BURN_IN, max(1, len(z) // 20))
    z_eval = clean_series(z[burn:])
    z_all = clean_series(z[1:])
    z2_eval = z_eval ** 2

    lb_sq_10 = safe_ljung_box(z2_eval, 10)
    lb_sq_20 = safe_ljung_box(z2_eval, 20)
    arch_10 = safe_arch_lm(z_eval, 10)

    std_z = z_eval.std(ddof=1)
    tail_ratio = float((z_eval.abs() > 3).mean())
    max_abs_z = float(z_eval.abs().max())

    passes_core = bool(
        opt.success
        and np.isfinite(loglik)
        and np.isfinite(std_z)
        and 0.60 <= std_z <= 1.40
        and tail_ratio <= 0.05
    )

    passes_sanity = bool(passes_core and max_abs_z <= 25)

    passes_vol = bool(
        np.isfinite(lb_sq_10)
        and np.isfinite(lb_sq_20)
        and np.isfinite(arch_10)
        and lb_sq_10 > 0.05
        and lb_sq_20 > 0.05
        and arch_10 > 0.05
    )

    summary = {
        "model": model_name,
        "x_cols": ", ".join(x_cols) if x_cols else "None",
        "n_obs": nobs,
        "loglik": loglik,
        "AIC": 2 * npar - 2 * loglik,
        "BIC": np.log(nobs) * npar - 2 * loglik,
        "num_params": npar,
        "success": bool(opt.success),
        "diagnostic_burn_in": burn,

        "std_resid_mean": float(z_eval.mean()),
        "std_resid_std": float(std_z),
        "std_resid_min": float(z_eval.min()),
        "std_resid_max": float(z_eval.max()),
        "max_abs_std_resid": max_abs_z,
        "abs_std_resid_gt_3": int((z_eval.abs() > 3).sum()),
        "abs_std_resid_gt_3_ratio": tail_ratio,

        "full_sample_max_abs_std_resid": float(z_all.abs().max()),
        "has_extreme_outlier_full_sample": bool(z_all.abs().max() > 25),

        "LB_resid_p_lag10": safe_ljung_box(z_eval, 10),
        "LB_resid_p_lag20": safe_ljung_box(z_eval, 20),
        "LB_sq_resid_p_lag10": lb_sq_10,
        "LB_sq_resid_p_lag20": lb_sq_20,
        "ARCH_LM_p_lag10": arch_10,

        "passes_core_sanity": passes_core,
        "passes_sanity": passes_sanity,
        "passes_volatility_diagnostics": passes_vol,
    }

    coef_table = coefficient_table(
        model_name=model_name,
        theta=theta,
        x_cols=x_cols,
        eps_arr=eps_arr,
        X_arr=X_arr,
        bounds=bounds,
        compute_pvalues=COMPUTE_PVALUES,
    )

    return {
        "model": model_name,
        "data": d,
        "x_cols": x_cols,
        "theta": theta,
        "summary": summary,
        "coef_table": coef_table,
        "h": h,
        "logh": logh,
        "z": z,
        "optimizer": opt,
    }


In [20]:
# ============================================================
# 5.5 MODEL SPECS
# ============================================================

signed_lag1 = [f"z_{c}_lag1" for c in SHOCK_BASE if f"z_{c}_lag1" in shock_df.columns]
abs_lag1 = [f"abs_z_{c}_lag1" for c in SHOCK_BASE if f"abs_z_{c}_lag1" in shock_df.columns]

crisis_term = (
    ["crisis_dummy"]
    if "crisis_dummy" in shock_df.columns and shock_df["crisis_dummy"].nunique(dropna=True) > 1
    else []
)

devaluation_term = (
    [DEVALUATION_DUMMY_COL]
    if DEVALUATION_DUMMY_COL in shock_df.columns and shock_df[DEVALUATION_DUMMY_COL].nunique(dropna=True) > 1
    else []
)

candidate_specs = {
    "ARMA32_EGARCH_t_no_X": [],
    "ARMA32_EGARCHX_t_devaluation_dummy_only": devaluation_term,
    "ARMA32_EGARCHX_t_signed_shocks_lag1": signed_lag1 + crisis_term,
    "ARMA32_EGARCHX_t_abs_shocks_lag1": abs_lag1 + crisis_term,
    "ARMA32_EGARCHX_t_signed_shocks_lag1_plus_devaluation": signed_lag1 + crisis_term + devaluation_term,
    "ARMA32_EGARCHX_t_abs_shocks_lag1_plus_devaluation": abs_lag1 + crisis_term + devaluation_term,
}

EGARCHX_SPECS = {}
seen = set()

for name, cols in candidate_specs.items():
    cols = list(dict.fromkeys(cols))
    key = tuple(cols)

    if key not in seen:
        EGARCHX_SPECS[name] = cols
        seen.add(key)

pd.Series(EGARCHX_SPECS, name="x_cols")


ARMA32_EGARCH_t_no_X                                                                                   []
ARMA32_EGARCHX_t_devaluation_dummy_only                                      [devaluation_20100211_dummy]
ARMA32_EGARCHX_t_signed_shocks_lag1                     [z_vix_change_lag1, z_us10y_change_lag1, z_dxy...
ARMA32_EGARCHX_t_abs_shocks_lag1                        [abs_z_vix_change_lag1, abs_z_us10y_change_lag...
ARMA32_EGARCHX_t_signed_shocks_lag1_plus_devaluation    [z_vix_change_lag1, z_us10y_change_lag1, z_dxy...
ARMA32_EGARCHX_t_abs_shocks_lag1_plus_devaluation       [abs_z_vix_change_lag1, abs_z_us10y_change_lag...
Name: x_cols, dtype: object

In [21]:
# ============================================================
# 5.6 RUN MODELS
# ============================================================

egarchx_results = {}

for name, x_cols in EGARCHX_SPECS.items():
    print("Fitting:", name)
    egarchx_results[name] = fit_egarchx(
        data=shock_df,
        y_col="arma_resid",
        x_cols=x_cols,
        model_name=name,
        maxiter=MAXITER,
    )

egarchx_summary = (
    pd.DataFrame([r["summary"] for r in egarchx_results.values()])
    .sort_values(
        ["passes_sanity", "passes_volatility_diagnostics", "BIC", "AIC"],
        ascending=[False, False, True, True],
        na_position="last",
    )
    .reset_index(drop=True)
)

egarchx_coef_table = pd.concat(
    [r["coef_table"] for r in egarchx_results.values()],
    ignore_index=True
)

display(egarchx_summary)
display(egarchx_coef_table)


Fitting: ARMA32_EGARCH_t_no_X
Fitting: ARMA32_EGARCHX_t_devaluation_dummy_only
Fitting: ARMA32_EGARCHX_t_signed_shocks_lag1
Fitting: ARMA32_EGARCHX_t_abs_shocks_lag1
Fitting: ARMA32_EGARCHX_t_signed_shocks_lag1_plus_devaluation
Fitting: ARMA32_EGARCHX_t_abs_shocks_lag1_plus_devaluation


,model,x_cols,n_obs,loglik,AIC,BIC,num_params,success,diagnostic_burn_in,std_resid_mean,std_resid_std,std_resid_min,std_resid_max,max_abs_std_resid,abs_std_resid_gt_3,abs_std_resid_gt_3_ratio,full_sample_max_abs_std_resid,has_extreme_outlier_full_sample,LB_resid_p_lag10,LB_resid_p_lag20,LB_sq_resid_p_lag10,LB_sq_resid_p_lag20,ARCH_LM_p_lag10,passes_core_sanity,passes_sanity,passes_volatility_diagnostics
0,ARMA32_EGARCHX_t_devaluation_dummy_only,devaluation_20100211_dummy,3997,"3,350.247409","-6,688.494818","-6,650.735022",6,True,60,-0.055323,0.947480,-8.067944,16.699534,16.699534,48,0.012189,16.699534,False,0.000009,0.000011,0.994958,0.999788,0.994477,True,True,True
1,ARMA32_EGARCH_t_no_X,None,3997,"3,342.486788","-6,674.973577","-6,643.507080",5,True,60,-0.055022,0.968892,-7.930780,16.823736,16.823736,51,0.012951,20.610148,False,0.000003,0.000004,0.979600,0.998977,0.979986,True,True,True
2,ARMA32_EGARCHX_t_abs_shocks_lag1_plus_devaluation,"abs_z_vix_change_lag1, abs_z_us10y_change_lag1...",3996,"3,360.575560","-6,697.151120","-6,621.634530",12,True,60,-0.057833,0.940792,-8.268327,14.669000,14.669000,46,0.011684,14.669000,False,0.000001,0.000000,0.981353,0.995954,0.979428,True,True,True
3,ARMA32_EGARCHX_t_abs_shocks_lag1,"abs_z_vix_change_lag1, abs_z_us10y_change_lag1...",3996,"3,353.075033","-6,684.150066","-6,614.926525",11,True,60,-0.057227,0.955413,-8.129509,14.812412,14.812412,50,0.012700,20.078309,False,0.000000,0.000000,0.925742,0.987194,0.926229,True,True,True
4,ARMA32_EGARCHX_t_signed_shocks_lag1_plus_deval...,"z_vix_change_lag1, z_us10y_change_lag1, z_dxy_...",3996,"3,355.497253","-6,686.994506","-6,611.477916",12,True,60,-0.056701,0.941820,-8.004589,16.447509,16.447509,46,0.011684,16.447509,False,0.000001,0.000000,0.994653,0.999791,0.994277,True,True,True
5,ARMA32_EGARCHX_t_signed_shocks_lag1,"z_vix_change_lag1, z_us10y_change_lag1, z_dxy_...",3996,"3,349.025252","-6,676.050505","-6,606.826964",11,True,60,-0.057469,0.958234,-7.935418,16.535642,16.535642,48,0.012192,19.847810,False,0.000000,0.000000,0.974139,0.998796,0.974886,True,True,True


,model,parameter,coef,std_error,t_stat,p_value,significant_10pct,significant_5pct,significant_1pct,at_bound
0,ARMA32_EGARCH_t_no_X,omega,-0.144470,0.023560,-6.132011,0.000000,True,True,True,False
1,ARMA32_EGARCH_t_no_X,alpha_abs_shock,0.472299,0.032821,14.390336,0.000000,True,True,True,False
2,ARMA32_EGARCH_t_no_X,gamma_asymmetry,0.009919,0.019742,0.502417,0.615374,False,False,False,False
3,ARMA32_EGARCH_t_no_X,beta_log_variance,0.964411,0.005388,178.993488,0.000000,True,True,True,False
4,ARMA32_EGARCH_t_no_X,nu_student_t,2.829821,0.140646,20.120195,0.000000,True,True,True,False
5,ARMA32_EGARCHX_t_devaluation_dummy_only,omega,-0.164402,0.027304,-6.021065,0.000000,True,True,True,False
6,ARMA32_EGARCHX_t_devaluation_dummy_only,alpha_abs_shock,0.531084,0.043825,12.118416,0.000000,True,True,True,False
7,ARMA32_EGARCHX_t_devaluation_dummy_only,gamma_asymmetry,0.017248,0.021070,0.818621,0.413003,False,False,False,False
8,ARMA32_EGARCHX_t_devaluation_dummy_only,beta_log_variance,0.958214,0.006393,149.879819,0.000000,True,True,True,False
9,ARMA32_EGARCHX_t_devaluation_dummy_only,nu_student_t,2.780919,0.145771,19.077269,0.000000,True,True,True,False


In [22]:
# ============================================================
# 5.7 EXTREME RESIDUAL TABLE AND MODEL SELECTION
# ============================================================

def inspect_extreme_days(result, top_n=10):
    d = result["data"].copy().reset_index(drop=True)

    d["model"] = result["model"]
    d["obs_no"] = np.arange(len(d))
    d["used_in_diagnostics"] = d["obs_no"] >= result["summary"]["diagnostic_burn_in"]
    d["egarchx_cond_vol"] = np.sqrt(result["h"])
    d["egarchx_std_resid"] = result["z"]
    d["abs_egarchx_std_resid"] = d["egarchx_std_resid"].abs()

    cols = [
        "model",
        "obs_no",
        "used_in_diagnostics",
        "Date",
        "arma_resid",
        "egarchx_cond_vol",
        "egarchx_std_resid",
        "abs_egarchx_std_resid",
        DEVALUATION_DUMMY_COL,
        "crisis_dummy",
    ] + SHOCK_BASE

    cols = [c for c in cols if c in d.columns]

    return (
        d.sort_values("abs_egarchx_std_resid", ascending=False)
        [cols]
        .head(top_n)
    )


egarchx_extreme_table = pd.concat(
    [inspect_extreme_days(r, top_n=10) for r in egarchx_results.values()],
    ignore_index=True
)

strict_pool = egarchx_summary[
    egarchx_summary["passes_sanity"]
    & egarchx_summary["passes_volatility_diagnostics"]
    & np.isfinite(egarchx_summary["BIC"])
]

core_pool = egarchx_summary[
    egarchx_summary["passes_core_sanity"]
    & egarchx_summary["passes_volatility_diagnostics"]
    & np.isfinite(egarchx_summary["BIC"])
]

if len(strict_pool):
    selection_pool = strict_pool
    best_egarchx_selection_rule = "strict_post_burnin_sanity"
elif len(core_pool):
    selection_pool = core_pool
    best_egarchx_selection_rule = "core_sanity_only"
else:
    selection_pool = pd.DataFrame()
    best_egarchx_selection_rule = "none"

if len(selection_pool):
    best_egarchx_name = selection_pool.sort_values(["BIC", "AIC"]).iloc[0]["model"]
    best_egarchx = egarchx_results[best_egarchx_name]
else:
    best_egarchx_name = None
    best_egarchx = None

egarchx_summary.to_csv(OUT_DIR / "table_3_custom_egarchx_model_comparison.csv", index=False)
egarchx_coef_table.to_csv(OUT_DIR / "table_4_custom_egarchx_coefficients_with_pvalues.csv", index=False)
egarchx_extreme_table.to_csv(OUT_DIR / "table_4b_custom_egarchx_extreme_residuals.csv", index=False)

print("Best custom EGARCH-X model:", best_egarchx_name)
print("Selection rule:", best_egarchx_selection_rule)

if best_egarchx is not None:
    best_coef_bound = best_egarchx["coef_table"].query("at_bound == True").copy()

    if len(best_coef_bound):
        print(
            "Parameter bound warning: at least one selected EGARCH-X coefficient is at an optimizer bound; "
            "do not over-interpret its exact magnitude or p-value."
        )
        display(best_coef_bound[["model", "parameter", "coef", "p_value", "at_bound"]])

    best_diag = egarchx_summary.loc[egarchx_summary["model"].eq(best_egarchx_name)].iloc[0]

    if (
        best_diag["passes_volatility_diagnostics"]
        and (
            best_diag["LB_resid_p_lag10"] <= 0.05
            or best_diag["LB_resid_p_lag20"] <= 0.05
        )
    ):
        print(
            "Diagnostic note: the selected EGARCH-X model passes volatility diagnostics, "
            "but standardized residual autocorrelation remains. Do not claim full diagnostics pass."
        )


display(egarchx_extreme_table.head(30))


Best custom EGARCH-X model: ARMA32_EGARCHX_t_devaluation_dummy_only
Selection rule: strict_post_burnin_sanity
Parameter bound warning: at least one selected EGARCH-X coefficient is at an optimizer bound; do not over-interpret its exact magnitude or p-value.


,model,parameter,coef,p_value,at_bound
5,ARMA32_EGARCHX_t_devaluation_dummy_only,delta_devaluation_20100211_dummy,3.000000,NaN,True


Diagnostic note: the selected EGARCH-X model passes volatility diagnostics, but standardized residual autocorrelation remains. Do not claim full diagnostics pass.


,model,obs_no,used_in_diagnostics,Date,arma_resid,egarchx_cond_vol,egarchx_std_resid,abs_egarchx_std_resid,devaluation_20100211_dummy,crisis_dummy,vix_change,us10y_change,dxy_return,oil_return,vnindex_return
0,ARMA32_EGARCH_t_no_X,6,False,2010-02-11,2.009600,0.097505,20.610148,20.610148,1,1,-1.440000,0.010000,-0.049995,1.001945,1.822728
1,ARMA32_EGARCH_t_no_X,258,True,2011-02-11,0.789300,0.046916,16.823736,16.823736,0,1,-0.400000,-0.060000,0.268010,-1.236546,-0.048067
2,ARMA32_EGARCH_t_no_X,2126,True,2018-07-23,0.677004,0.060570,11.177203,11.177203,0,0,-0.240000,0.070000,0.222069,-3.487800,0.358264
3,ARMA32_EGARCH_t_no_X,1386,True,2015-08-12,0.834309,0.081731,10.207975,10.207975,0,0,-0.100000,-0.010000,-1.064333,0.254836,-1.447503
4,ARMA32_EGARCH_t_no_X,2317,True,2019-04-26,0.298054,0.036174,8.239580,8.239580,0,0,-0.520000,-0.030000,-0.193665,-3.019215,0.564039
5,ARMA32_EGARCH_t_no_X,308,True,2011-04-26,-1.189459,0.149980,-7.930780,7.930780,0,0,-0.150000,-0.050000,-0.202938,0.035810,-0.810804
6,ARMA32_EGARCH_t_no_X,2953,True,2021-11-04,-0.296246,0.042213,-7.017846,7.017846,0,0,0.340000,-0.070000,0.520694,-2.429675,0.279330
7,ARMA32_EGARCH_t_no_X,3082,True,2022-05-11,0.470459,0.067406,6.979517,6.979517,0,1,-0.430000,-0.080000,-0.067382,5.614415,0.614239
8,ARMA32_EGARCH_t_no_X,136,True,2010-08-18,1.739807,0.263073,6.613396,6.613396,0,1,0.260000,0.000000,-0.012164,-0.489581,-1.747577
9,ARMA32_EGARCH_t_no_X,1427,True,2015-10-09,-1.008773,0.162158,-6.220927,6.220927,0,0,-0.340000,0.000000,-0.536479,0.423687,0.211100


Robustness / event-adjusted model:
ARMA(3,2)-EGARCH-X with the fixed `2010-02-11` devaluation dummy.

Interpretation constraints:
- The dummy is a known policy-event dummy, not a variable mechanically selected from the largest ARMA residual.
- If the devaluation coefficient is at the imposed optimizer bound, interpret it as evidence of a very large policy-related volatility spike; do not interpret the exact coefficient magnitude literally.
- The selected EGARCH-X model is interpreted primarily as a volatility model. Passing volatility diagnostics does not mean all standardized-residual autocorrelation diagnostics pass.


## 6. Supplementary HAC Shock-Volatility Regression

This is not called EGARCH-X. It is a two-step explanatory regression using estimated EGARCH volatility.
Use this section for coefficient significance and easy interpretation.

In [23]:
# ============================================================
# 6. SUPPLEMENTARY HAC SHOCK-VOLATILITY REGRESSIONS
# Purpose: explain selected conditional volatility
# ============================================================

import statsmodels.api as sm

# ------------------------------------------------------------
# 6.1 Get selected conditional volatility from best EGARCH-X
# ------------------------------------------------------------

if best_egarchx is None:
    raise ValueError("best_egarchx is None. Run and select EGARCH-X model first.")

vol_df = best_egarchx["data"].copy()
vol_df["log_selected_cond_vol"] = 0.5 * np.log(best_egarchx["h"])

hac_df = shock_df.copy()

if "Date" in hac_df.columns and "Date" in vol_df.columns:
    hac_df["Date"] = pd.to_datetime(hac_df["Date"])
    vol_df["Date"] = pd.to_datetime(vol_df["Date"])

    hac_df = hac_df.drop(columns=["log_selected_cond_vol"], errors="ignore")
    hac_df = hac_df.merge(
        vol_df[["Date", "log_selected_cond_vol"]],
        on="Date",
        how="left"
    )
else:
    hac_df["log_selected_cond_vol"] = np.nan
    hac_df.loc[vol_df.index, "log_selected_cond_vol"] = vol_df["log_selected_cond_vol"]


# ------------------------------------------------------------
# 6.2 Keep only useful HAC specifications
# ------------------------------------------------------------

signed_lag1 = [
    f"z_{c}_lag1"
    for c in shock_base
    if f"z_{c}_lag1" in hac_df.columns
]

abs_lag1 = [
    f"abs_z_{c}_lag1"
    for c in shock_base
    if f"abs_z_{c}_lag1" in hac_df.columns
]

controls = []

if "crisis_dummy" in hac_df.columns and hac_df["crisis_dummy"].nunique(dropna=True) > 1:
    controls.append("crisis_dummy")

if DEVALUATION_DUMMY_COL in hac_df.columns and hac_df[DEVALUATION_DUMMY_COL].nunique(dropna=True) > 1:
    controls.append(DEVALUATION_DUMMY_COL)

hac_specs = {
    "HAC_signed_lag1": signed_lag1 + controls,
    "HAC_abs_lag1": abs_lag1 + controls,
}


# ------------------------------------------------------------
# 6.3 HAC regression helper
# ------------------------------------------------------------

def run_hac_regression(data, y_col, x_cols, model_name, hac_lags=5):
    cols = [y_col] + x_cols
    d = data[cols].replace([np.inf, -np.inf], np.nan).dropna()

    y = d[y_col].astype(float)
    X = sm.add_constant(d[x_cols].astype(float), has_constant="add")

    res = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": hac_lags}
    )

    coef_table = pd.DataFrame({
        "model": model_name,
        "variable": res.params.index,
        "coef": res.params.values,
        "std_error_HAC": res.bse.values,
        "t_stat": res.tvalues.values,
        "p_value": res.pvalues.values,
    })

    coef_table["significance"] = np.select(
        [
            coef_table["p_value"] < 0.01,
            coef_table["p_value"] < 0.05,
            coef_table["p_value"] < 0.10,
        ],
        ["***", "**", "*"],
        default="",
    )

    summary_row = {
        "model": model_name,
        "nobs": int(res.nobs),
        "r2": res.rsquared,
        "adj_r2": res.rsquared_adj,
        "AIC": res.aic,
        "BIC": res.bic,
        "hac_lags": hac_lags,
        "x_cols": ", ".join(x_cols),
    }

    return res, coef_table, summary_row


# ------------------------------------------------------------
# 6.4 Run HAC models
# ------------------------------------------------------------

hac_models = {}
hac_coef_tables = []
hac_summary_rows = []

for model_name, x_cols in hac_specs.items():
    x_cols = [c for c in x_cols if c in hac_df.columns]

    res, coef_table, summary_row = run_hac_regression(
        data=hac_df,
        y_col="log_selected_cond_vol",
        x_cols=x_cols,
        model_name=model_name,
        hac_lags=5,
    )

    hac_models[model_name] = res
    hac_coef_tables.append(coef_table)
    hac_summary_rows.append(summary_row)

hac_coef_table = pd.concat(hac_coef_tables, ignore_index=True)
hac_summary = (
    pd.DataFrame(hac_summary_rows)
    .sort_values(["BIC", "AIC"])
    .reset_index(drop=True)
)

hac_coef_table.to_csv(
    OUT_DIR / "table_5_hac_shock_volatility_coefficients.csv",
    index=False
)

hac_summary.to_csv(
    OUT_DIR / "table_6_hac_shock_volatility_model_summary.csv",
    index=False
)

display(hac_summary)
display(hac_coef_table)

,model,nobs,r2,adj_r2,AIC,BIC,hac_lags,x_cols
0,HAC_abs_lag1,3997,0.034971,0.033278,"7,999.469260","8,049.815655",5,"abs_z_vix_change_lag1, abs_z_us10y_change_lag1..."
1,HAC_signed_lag1,3997,0.005359,0.003613,"8,120.274956","8,170.621351",5,"z_vix_change_lag1, z_us10y_change_lag1, z_dxy_..."


,model,variable,coef,std_error_HAC,t_stat,p_value,significance
0,HAC_signed_lag1,const,-1.990395,0.024576,-80.990706,0.000000,***
1,HAC_signed_lag1,z_vix_change_lag1,-0.006124,0.009868,-0.620530,0.534909,
2,HAC_signed_lag1,z_us10y_change_lag1,-0.011627,0.010623,-1.094511,0.273731,
3,HAC_signed_lag1,z_dxy_return_lag1,-0.015880,0.010264,-1.547209,0.121813,
4,HAC_signed_lag1,z_oil_return_lag1,-0.003770,0.009314,-0.404720,0.685683,
5,HAC_signed_lag1,z_vnindex_return_lag1,-0.020947,0.013164,-1.591237,0.111556,
6,HAC_signed_lag1,crisis_dummy,-0.096358,0.087430,-1.102120,0.270409,
7,HAC_signed_lag1,devaluation_20100211_dummy,1.247468,0.086483,14.424427,0.000000,***
8,HAC_abs_lag1,const,-2.183276,0.036676,-59.528324,0.000000,***
9,HAC_abs_lag1,abs_z_vix_change_lag1,-0.023728,0.020512,-1.156777,0.247363,


### Interpretation of HAC Absolute Shock Regression

| Variable | Coef | Interpretation |
|---|---:|---|
| `abs_z_us10y_change_lag1` | 0.0905| Lagged absolute changes in the US 10-year yield increase USD/VND volatility. |
| `abs_z_dxy_return_lag1` | 0.0892 | Lagged absolute DXY movements increase USD/VND volatility. |
| `abs_z_oil_return_lag1` | 0.0675 | Lagged oil price volatility increases USD/VND volatility. |
| `abs_z_vnindex_return_lag1` | 0.0838| Lagged VN-Index volatility increases USD/VND volatility. |
| `devaluation_20100211_dummy` | 0.4384 | The 2010-02-11 devaluation event strongly increases exchange-rate volatility. |
| `abs_z_vix_change_lag1` | -0.0443 | The coefficient is significant but negative, so it should be interpreted cautiously. |

## 7. Supplementary Weekly SVAR

This section estimates a structural VAR, not just a reduced-form VAR.

Identification strategy:

- Weekly aggregation is used to reduce daily noise and make dynamic transmission easier to interpret.
- All variables are standardized before estimation, so structural IRFs are comparable across variables.
- A recursive short-run A-model is imposed:
  - Variables earlier in the ordering can contemporaneously affect variables later in the ordering.
  - Variables later in the ordering cannot contemporaneously affect variables earlier in the same week.
- The ordering must be reported because recursive SVAR results depend on this identifying assumption.

Baseline ordering:

`vix_change -> us10y_change -> dxy_return -> oil_return -> vnindex_return -> target`

where `target` is either `fx_return` or `log_selected_cond_vol`.

In [24]:
# ============================================================
# 7. MINIMAL WEEKLY SVAR
# Flow: best_egarchx -> selected volatility -> weekly SVAR
# ============================================================

import numpy as np
import pandas as pd
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.svar_model import SVAR
from IPython.display import display


# ============================================================
# 7.1 BUILD WEEKLY DATA
# ============================================================

if best_egarchx is None:
    raise ValueError("best_egarchx is None. Run EGARCH-X first.")

var_df = df_model.copy()
var_df["Date"] = pd.to_datetime(var_df["Date"])

# Remove old / duplicated volatility columns from previous failed runs
var_df = var_df.drop(
    columns=[
        "log_egarch_vol",
        "log_selected_cond_vol",
        "log_selected_cond_vol_x",
        "log_selected_cond_vol_y",
    ],
    errors="ignore"
)

# Create selected volatility from best EGARCH-X
eg_data = best_egarchx["data"].copy()
log_vol = 0.5 * np.log(np.asarray(best_egarchx["h"]))

# Prefer Date merge if possible
if "Date" in eg_data.columns and len(eg_data) == len(log_vol):
    eg_data["Date"] = pd.to_datetime(eg_data["Date"])

    vol_df = pd.DataFrame({
        "Date": eg_data["Date"].values,
        "log_selected_cond_vol": log_vol,
    })

    var_df = var_df.merge(vol_df, on="Date", how="left")

# Fallback: align by original index
else:
    var_df["log_selected_cond_vol"] = np.nan
    var_df.loc[eg_data.index, "log_selected_cond_vol"] = log_vol

# Hard check
if "log_selected_cond_vol" not in var_df.columns:
    raise KeyError("log_selected_cond_vol was not created.")

if var_df["log_selected_cond_vol"].notna().sum() == 0:
    raise ValueError("log_selected_cond_vol was created but all values are NaN.")

var_df = var_df.set_index("Date").sort_index()

svar_shocks = [
    c for c in ["vix_change", "us10y_change", "dxy_return", "oil_return", "vnindex_return"]
    if c in var_df.columns
]

weekly = pd.DataFrame(index=var_df.resample("W-FRI").size().index)

for col in svar_shocks + ["fx_return"]:
    weekly[col] = var_df[col].resample("W-FRI").sum()

weekly["log_selected_cond_vol"] = (
    var_df["log_selected_cond_vol"]
    .resample("W-FRI")
    .mean()
)

weekly = weekly.replace([np.inf, -np.inf], np.nan).dropna()

weekly_z = (weekly - weekly.mean()) / weekly.std(ddof=0)
weekly_z = weekly_z.replace([np.inf, -np.inf], np.nan).dropna()

print("Weekly SVAR sample size:", len(weekly_z))
print("Weekly SVAR columns:", list(weekly_z.columns))

Weekly SVAR sample size: 831
Weekly SVAR columns: ['vix_change', 'us10y_change', 'dxy_return', 'oil_return', 'vnindex_return', 'fx_return', 'log_selected_cond_vol']


In [25]:
# ============================================================
# 7.2 MINIMAL SVAR FUNCTIONS
# ============================================================

def recursive_A(k):
    A = np.empty((k, k), dtype=object)

    for i in range(k):
        for j in range(k):
            if i == j:
                A[i, j] = 1.0
            elif i > j:
                A[i, j] = "E"
            else:
                A[i, j] = 0.0

    return A


def choose_lag(d, maxlags=6):
    maxlags = min(maxlags, max(1, len(d) // 10))

    selected = VAR(d).select_order(maxlags=maxlags).selected_orders

    lag = selected.get("bic")
    if lag is None or lag < 1:
        lag = selected.get("aic")
    if lag is None or lag < 1:
        lag = 1

    return int(lag)


def run_svar(data, variables, target, model_name, horizon=12):
    variables = [v for v in variables if v in data.columns]
    d = data[variables].dropna()

    if target not in variables:
        raise ValueError(f"{target} is not in variables.")

    if len(d) < 80:
        raise ValueError(f"Too few weekly observations for {model_name}.")

    lag = choose_lag(d)

    A = recursive_A(len(variables))
    A_guess = np.full(int(np.sum(A == "E")), 0.05)

    model = SVAR(d, svar_type="A", A=A)

    try:
        res = model.fit(
            A_guess=A_guess,
            maxlags=lag,
            trend="c",
            solver="bfgs",
            maxiter=1000,
            override=False,
        )
        fit_status = "OK"
    except Exception:
        res = model.fit(
            A_guess=A_guess,
            maxlags=lag,
            trend="c",
            solver="powell",
            maxiter=2000,
            override=False,
        )
        fit_status = "OK_powell"

    granger_rows = []

    for shock in variables:
        if shock == target:
            continue

        try:
            test = res.test_causality(
                caused=target,
                causing=[shock],
                kind="f"
            )

            granger_rows.append({
                "model": model_name,
                "target": target,
                "shock": shock,
                "lag": lag,
                "p_value": test.pvalue,
                "significant_5pct": test.pvalue < 0.05,
            })

        except Exception as e:
            granger_rows.append({
                "model": model_name,
                "target": target,
                "shock": shock,
                "lag": lag,
                "p_value": np.nan,
                "significant_5pct": False,
                "error": str(e)[:120],
            })

    irf = res.irf(horizon).svar_irfs
    target_idx = variables.index(target)

    irf_rows = []

    for shock in variables:
        shock_idx = variables.index(shock)

        for h in range(horizon + 1):
            irf_rows.append({
                "model": model_name,
                "target": target,
                "shock": shock,
                "horizon_weeks": h,
                "structural_irf": irf[h, target_idx, shock_idx],
            })

    summary = pd.DataFrame([{
        "model": model_name,
        "target": target,
        "n_obs_weekly": len(d),
        "n_variables": len(variables),
        "lag": lag,
        "ordering": " -> ".join(variables),
        "fit_status": fit_status,
    }])

    return summary, pd.DataFrame(granger_rows), pd.DataFrame(irf_rows)

In [26]:
# ============================================================
# 7.3 RUN AND EXPORT
# ============================================================

svar_specs = {
    "SVAR_shocks_to_fx_return": {
        "variables": svar_shocks + ["fx_return"],
        "target": "fx_return",
    },
    "SVAR_shocks_to_selected_volatility": {
        "variables": svar_shocks + ["log_selected_cond_vol"],
        "target": "log_selected_cond_vol",
    },
}

svar_results = []
svar_errors = []

for name, spec in svar_specs.items():
    try:
        svar_results.append(
            run_svar(
                data=weekly_z,
                variables=spec["variables"],
                target=spec["target"],
                model_name=name,
                horizon=12,
            )
        )
    except Exception as e:
        svar_errors.append({
            "model": name,
            "error": str(e)[:300],
        })

if svar_results:
    svar_summary = pd.concat([x[0] for x in svar_results], ignore_index=True)
    svar_granger = pd.concat([x[1] for x in svar_results], ignore_index=True)
    svar_structural_irf = pd.concat([x[2] for x in svar_results], ignore_index=True)
else:
    svar_summary = pd.DataFrame()
    svar_granger = pd.DataFrame()
    svar_structural_irf = pd.DataFrame()

svar_errors = pd.DataFrame(svar_errors)

svar_summary.to_csv(OUT_DIR / "table_7_weekly_svar_summary.csv", index=False)
svar_granger.to_csv(OUT_DIR / "table_8_weekly_svar_granger_causality.csv", index=False)
svar_structural_irf.to_csv(OUT_DIR / "table_9_weekly_svar_structural_irf.csv", index=False)
svar_errors.to_csv(OUT_DIR / "table_10_weekly_svar_errors.csv", index=False)

display(svar_summary)
display(svar_granger)
display(svar_structural_irf.head())

if len(svar_errors):
    display(svar_errors)

,model,target,n_obs_weekly,n_variables,lag,ordering,fit_status
0,SVAR_shocks_to_fx_return,fx_return,831,6,4,vix_change -> us10y_change -> dxy_return -> oi...,OK
1,SVAR_shocks_to_selected_volatility,log_selected_cond_vol,831,6,1,vix_change -> us10y_change -> dxy_return -> oi...,OK


,model,target,shock,lag,p_value,significant_5pct
0,SVAR_shocks_to_fx_return,fx_return,vix_change,4,0.139520,False
1,SVAR_shocks_to_fx_return,fx_return,us10y_change,4,0.000015,True
2,SVAR_shocks_to_fx_return,fx_return,dxy_return,4,0.097951,False
3,SVAR_shocks_to_fx_return,fx_return,oil_return,4,0.316453,False
4,SVAR_shocks_to_fx_return,fx_return,vnindex_return,4,0.185996,False
5,SVAR_shocks_to_selected_volatility,log_selected_cond_vol,vix_change,1,0.844721,False
6,SVAR_shocks_to_selected_volatility,log_selected_cond_vol,us10y_change,1,0.245552,False
7,SVAR_shocks_to_selected_volatility,log_selected_cond_vol,dxy_return,1,0.525032,False
8,SVAR_shocks_to_selected_volatility,log_selected_cond_vol,oil_return,1,0.026551,True
9,SVAR_shocks_to_selected_volatility,log_selected_cond_vol,vnindex_return,1,0.639268,False


,model,target,shock,horizon_weeks,structural_irf
0,SVAR_shocks_to_fx_return,fx_return,vix_change,0,0.085932
1,SVAR_shocks_to_fx_return,fx_return,vix_change,1,0.043779
2,SVAR_shocks_to_fx_return,fx_return,vix_change,2,0.056803
3,SVAR_shocks_to_fx_return,fx_return,vix_change,3,0.039042
4,SVAR_shocks_to_fx_return,fx_return,vix_change,4,0.048096


## 8. Final Separated Comparison Tables

This section saves separate tables. Do not sort ARMA, GARCH, EGARCH-X, HAC, and SVAR in one common ranking table because they answer different questions.

In [27]:
# ============================================================
# 8. FINAL SEPARATED OUTPUT SUMMARY
# ============================================================

final_summary_rows = []

final_summary_rows.append({
    "block": "ARMA mean baseline",
    "selected_model": best_arma_name,
    "selection_rule": "Lowest BIC among converged ARMA(p,q) models",
    "output_file": "table_1_arma_mean_selection.csv",
})

final_summary_rows.append({
    "block": "GARCH-family volatility",
    "selected_model": best_garch_name,
    "selection_rule": "Lowest BIC among converged models passing residual sanity checks",
    "output_file": "table_2_garch_family_comparison_and_diagnostics.csv",
})

final_summary_rows.append({
    "block": "True EGARCH-X variance equation",
    "selected_model": best_egarchx_name if best_egarchx_name is not None else "No EGARCH-X model passed sanity checks",
    "selection_rule": "Lowest BIC among custom EGARCH/EGARCH-X models passing sanity checks",
    "output_file": "table_3_custom_egarchx_model_comparison.csv; table_4_custom_egarchx_coefficients_with_pvalues.csv; table_4b_custom_egarchx_extreme_residuals.csv",
})

if len(hac_summary):
    best_hac_model = hac_summary.dropna(subset=["BIC"]).sort_values(["BIC", "AIC"]).iloc[0]["model"] if hac_summary["BIC"].notna().any() else "No valid HAC model"
else:
    best_hac_model = "No valid HAC model"

final_summary_rows.append({
    "block": "Supplementary HAC shock-volatility regression",
    "selected_model": best_hac_model,
    "selection_rule": "Lowest BIC, used for supplementary interpretation only",
    "output_file": "table_5_hac_shock_volatility_coefficients.csv; table_6_hac_shock_volatility_model_summary.csv",
})

final_summary_rows.append({
    "block": "Supplementary weekly SVAR",
    "selected_model": "Weekly recursive short-run A-model SVAR",
    "selection_rule": "Lag selected by BIC, fallback to AIC, minimum lag 1; structural identification through recursive A restrictions",
    "output_file": "table_7_weekly_svar_summary.csv; table_8_weekly_svar_granger_causality.csv; table_9_weekly_svar_structural_irf.csv; table_10_weekly_svar_errors.csv",
})

final_summary = pd.DataFrame(final_summary_rows)
final_summary.to_csv(OUT_DIR / "final_modeling_summary.csv", index=False)

display(final_summary)

print("\nFiles saved in:", OUT_DIR.resolve())
for p in sorted(OUT_DIR.glob("*.csv")):
    print("-", p.name)

,block,selected_model,selection_rule,output_file
0,ARMA mean baseline,"ARMA(3,2)","Lowest BIC among converged ARMA(p,q) models",table_1_arma_mean_selection.csv
1,GARCH-family volatility,"ARMA(3,2)-EGARCH(1,1)-t",Lowest BIC among converged models passing resi...,table_2_garch_family_comparison_and_diagnostic...
2,True EGARCH-X variance equation,ARMA32_EGARCHX_t_devaluation_dummy_only,Lowest BIC among custom EGARCH/EGARCH-X models...,table_3_custom_egarchx_model_comparison.csv; t...
3,Supplementary HAC shock-volatility regression,HAC_abs_lag1,"Lowest BIC, used for supplementary interpretat...",table_5_hac_shock_volatility_coefficients.csv;...
4,Supplementary weekly SVAR,Weekly recursive short-run A-model SVAR,"Lag selected by BIC, fallback to AIC, minimum ...",table_7_weekly_svar_summary.csv; table_8_weekl...



Files saved in: /Users/klinhfhm/Documents/Seminar 6/Time Series/final/final-time-series/outputs/model_egarchx_hac_svar_final
- final_modeling_summary.csv
- model_data_with_selected_conditional_volatility.csv
- table_10_weekly_svar_errors.csv
- table_1_arma_mean_selection.csv
- table_2_garch_family_comparison_and_diagnostics.csv
- table_3_custom_egarchx_model_comparison.csv
- table_4_custom_egarchx_coefficients_with_pvalues.csv
- table_4b_custom_egarchx_extreme_residuals.csv
- table_5_hac_shock_volatility_coefficients.csv
- table_6_hac_shock_volatility_model_summary.csv
- table_7_weekly_svar_summary.csv
- table_8_weekly_svar_granger_causality.csv
- table_9_weekly_svar_structural_irf.csv


## 9. Interpretation Rules for the Report

Use these rules when writing the methodology and results sections:

1. ARMA is only the mean-equation baseline. Do not compare it directly against GARCH/EGARCH as if they were the same model class.
2. `arch_model(..., mean="AR", lags=1)` estimates AR(1)-GARCH/EGARCH, not ARMA-GARCH/EGARCH.
3. The custom EGARCH-X section is the only section where shock variables enter the variance equation directly.
4. The HAC section is a two-step explanatory regression, not EGARCH-X.
5. `vnindex_return` is a domestic control, not a global shock.
6. The weekly SVAR imposes recursive short-run A-model restrictions. Report the ordering and the A restrictions.
7. Structural IRFs depend on the recursive ordering. Interpret them as responses to identified structural shocks, not automatic proof of causality.
8. Do not interpret any model that fails convergence or sanity checks.